In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# Add paired IPEDS label columns using the value-label lookup
# Only process columns named in IPEDS_Value_Labels_Lookup.xlsx
# ============================================================

# ----------------------------
# File paths
# ----------------------------
input_file_path = Path("TitleII_IPEDS_Institution_Lookup.xlsx")
value_labels_file_path = Path("IPEDS_Value_Labels_Lookup.xlsx")
output_file_path = Path("TitleII_IPEDS_Institution_Lookup_With_Labels.xlsx")

# ----------------------------
# Load files
# ----------------------------
df_ipeds = pd.read_excel(input_file_path)
df_labels = pd.read_excel(value_labels_file_path)

# ----------------------------
# Clean lookup file headers only
# ----------------------------
df_labels.columns = df_labels.columns.astype(str).str.strip()

required_label_cols = ["VariableName", "Value", "ValueLabel"]
missing_required = [col for col in required_label_cols if col not in df_labels.columns]
if missing_required:
    raise KeyError(
        f"IPEDS_Value_Labels_Lookup is missing required columns: {missing_required}"
    )

# Strip lookup VariableName / ValueLabel only
df_labels["VariableName"] = df_labels["VariableName"].astype(str).str.strip()
df_labels["ValueLabel"] = df_labels["ValueLabel"].astype(str).str.strip()

# ----------------------------
# Helper to normalize coded values
# ----------------------------
def normalize_code(val):
    if pd.isna(val):
        return None

    if isinstance(val, str):
        v = val.strip()
        if v == "":
            return None
        try:
            num = float(v)
            if num.is_integer():
                return str(int(num))
            return str(num)
        except ValueError:
            return v

    if isinstance(val, (int, float)):
        if pd.isna(val):
            return None
        if float(val).is_integer():
            return str(int(val))
        return str(val)

    return str(val).strip()

# Normalize lookup values
df_labels["NormalizedValue"] = df_labels["Value"].apply(normalize_code)

# ----------------------------
# Build mappings by VariableName
# ----------------------------
label_mappings = {}
for variable_name, group in df_labels.groupby("VariableName", dropna=False):
    mapping = dict(zip(group["NormalizedValue"], group["ValueLabel"]))
    label_mappings[variable_name] = mapping

# ----------------------------
# Add paired label columns only for matching VariableName columns
# ----------------------------
matched_variables = []
missing_variables = []

for variable_name, mapping in label_mappings.items():
    # Find the exact matching column in df_ipeds, even if it has trailing spaces
    matching_cols = [
        col for col in df_ipeds.columns
        if str(col).strip() == variable_name
    ]

    if matching_cols:
        source_col = matching_cols[0]
        label_col = f"{variable_name}_Label"

        normalized_series = df_ipeds[source_col].apply(normalize_code)
        df_ipeds[label_col] = normalized_series.map(mapping)

        matched_variables.append(source_col)
    else:
        missing_variables.append(variable_name)

# ----------------------------
# Ensure ZIP and FIPS codes are zero-padded strings
# ----------------------------
zip_fips_columns = [
    "zip_code_TS",
    "Fips County code"
]

for col in zip_fips_columns:
    if col in df_ipeds.columns:
        df_ipeds[col] = (
            df_ipeds[col]
            .astype(str)
            .str.replace(r"\.0$", "", regex=True)  # remove Excel float artifacts
            .str.strip()
            .str.zfill(5)
        )

# ----------------------------
# Save output
# ----------------------------
df_ipeds.to_excel(output_file_path, index=False)

print(f"Enhanced dataset successfully saved to: {output_file_path}")

print("\nVariables for which label columns were added:")
for col in matched_variables:
    print(f" - {col}")

if missing_variables:
    print("\nWarning: These VariableName entries from the lookup file were not found in the institution lookup file:")
    for col in missing_variables:
        print(f" - {col}")

# ----------------------------
# Preview
# ----------------------------
preview_cols = [
    "Institution Name",
    "Control of institution",
    "Control of institution_Label",
    "Sector of institution",
    "Sector of institution_Label",
    "Status of institution",
    "Status of institution_Label",
]

existing_preview_cols = [col for col in preview_cols if col in df_ipeds.columns]

print("\nFirst 5 rows of the enhanced dataset:")
print(df_ipeds[existing_preview_cols].head())